# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the **FAIR\u00b2** dataset package using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

The dataset covers demographics, comorbidities, cancer types, treatment, MSI/MMR biomarker status, and more for 77 cancer survivors with second primary colorectal cancer. The notebook follows recommended best practices for referencing schema entities using their full `@id` in all operations.

### Dataset Source
Croissant JSON-LD Schema: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading
Load the dataset schema and metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object (use attribute access, not dict)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print("Publication Date:", metadata.datePublished)
print("Version:", metadata.version)
print("Identifier:", metadata.identifier)

# Display available record sets
print("\nRecord Set @ids:")
recset_ids = [rset['@id'] if isinstance(rset, dict) and '@id' in rset else rset for rset in getattr(metadata, 'recordSet', [])]
if recset_ids:
    for ridx, rid in enumerate(recset_ids):
        print(f"  {ridx+1:2d}. {rid}")
else:
    print("  (recordSet not embedded in schema metadata, searching in fields below)")

## 2. Data Overview
List available record sets, fields, and their `@id`s.

> Since the "recordSet" attribute array is empty in the metadata, let us programmatically fetch the available record set `@id`s from the Croissant schema via the loaded dataset object.

In [ ]:
# Find available record sets using dataset utility function

def list_record_sets(ds):
    """
    Utility: List available record set @ids from the dataset schema.
    """
    rset_ids = []
    try:
        # Try new API attribute
        rset_ids = ds.record_set_ids()
    except Exception:
        # Fallback: infer from records()
        attrs = getattr(ds, 'attributes', {})
        if hasattr(ds, 'record_sets'):
            rset_ids = list(ds.record_sets.keys())
    return rset_ids

record_set_ids = []
try:
    # mlcroissant >=0.4: Dataset.record_set_ids() exists
    record_set_ids = dataset.record_set_ids()
except Exception:
    # Fall back to inspecting records (if only one present, try with None)
    try:
        # Try to get first records, then access record_set key
        for rec in dataset.records():
            if hasattr(rec, '__record_set_id__'):
                record_set_ids.append(getattr(rec, '__record_set_id__'))
            elif hasattr(rec, '_record_set'):
                record_set_ids.append(getattr(rec, '_record_set'))
            # Only take first (avoid iterating entire set)
            break
        record_set_ids = list(set(record_set_ids))
    except Exception:
        pass
if not record_set_ids:
    # Try attribute
    if hasattr(dataset, 'record_sets'):
        record_set_ids = list(dataset.record_sets.keys())

# If still empty: hardcode the likely record set @id based on Croissant conventions
if not record_set_ids:
    # This is a sample form for many Croissant datasets
    record_set_ids = [
        "https://sen.science/doi/10.71728/senscience.qs2f-h81p/SecondPrimaryColorectalCancer"
    ]

# Display available record sets:
print("\nAVAILABLE RECORD SETS:")
for idx, rid in enumerate(record_set_ids):
    print(f"  {idx+1}. {rid}")

# For each record set, show a preview of field @ids and names
for record_set_id in record_set_ids:
    print(f"\n---\nFields in Record Set: {record_set_id}")
    try:
        # Fetch the schema for this record set
        record_set = dataset.record_set_schema(record_set_id)
        if record_set and hasattr(record_set, 'fields'):
            fields = getattr(record_set, 'fields')
            if isinstance(fields, dict):
                fields = [fields]
            for fld in fields:
                fid = getattr(fld, '@id', None) or fld.get('@id', None)
                fname = getattr(fld, 'name', None) or fld.get('name', None)
                print(f"    - {fid:70s} (name: {fname})")
        else:
            print("    (fields not available in record set schema)")
    except Exception as ex:
        print(f"    Could not load schema for {record_set_id}: {ex}")

## 3. Data Extraction
Load all data from each record set into a pandas DataFrame. You can refer to record sets and fields using their full Croissant `@id` string.

In [ ]:
# Collect all record sets into dataframes dict
dataframes = {}
for recset_id in record_set_ids:
    print(f"Loading records for record set: {recset_id}")
    records = list(dataset.records(record_set=recset_id))
    dataframes[recset_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records. Columns:", dataframes[recset_id].columns.tolist())

# For demonstration, choose the first record set
chosen_record_set = record_set_ids[0] if record_set_ids else None

if chosen_record_set and chosen_record_set in dataframes:
    print(f"\nSample data for record set: {chosen_record_set}")
    display(dataframes[chosen_record_set].head())
else:
    print("No record set data available for preview.")

## 4. Exploratory Data Analysis (EDA)

Filter and process the data using field `@id`s. We'll select a numeric field for some basic transformations and group the data using another field. Please refer to field `@id`s from the schema above. 

> **Tip:** If you don't yet know which columns are numeric, print `dataframes[chosen_record_set].dtypes` and inspect the table header for typical numerical columns (e.g., Age, DiagnosisIntervalMonths, etc.).

In [ ]:
# Inspect columns and dtypes
if chosen_record_set in dataframes:
    print("Columns for analysis:")
    print(dataframes[chosen_record_set].dtypes)
    print("\nExample row:")
    display(dataframes[chosen_record_set].head())

    # For example purposes, suppose '@id' for 'Age at Second Diagnosis' is:
    # 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/SecondPrimaryColorectalCancer/AgeAtSecondDiagnosis'
    # and for grouping by sex:
    # 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/SecondPrimaryColorectalCancer/Sex'
    numeric_field_id = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/SecondPrimaryColorectalCancer/AgeAtSecondDiagnosis'
    group_field_id = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/SecondPrimaryColorectalCancer/Sex'

    # Ensure expected columns present
    if numeric_field_id not in dataframes[chosen_record_set].columns:
        print(f"Column {numeric_field_id} not found in data.")
    else:
        # Numeric filtering
        threshold = 50
        filtered_df = dataframes[chosen_record_set][dataframes[chosen_record_set][numeric_field_id] > threshold].copy()
        print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
        print(filtered_df[[numeric_field_id]].head())

        # Normalization
        colname_norm = numeric_field_id + '_normalized'
        filtered_df[colname_norm] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized field for filtered records:")
        print(filtered_df[[numeric_field_id, colname_norm]].head())

        # Optional grouping (by, e.g., sex)
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nGrouped by {group_field_id}, mean {numeric_field_id}:")
            print(grouped_df.head())
else:
    print("No loaded dataframe to analyze.")

## 5. Visualization
Visualize the age distribution and show a boxplot versus sex. All references use the field `@id`.

> The plots below will use matplotlib and seaborn for quick exploratory visualizations.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Use field @ids
if chosen_record_set in dataframes:
    df = dataframes[chosen_record_set]
    # Replace with your actual field ids
    numeric_field = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/SecondPrimaryColorectalCancer/AgeAtSecondDiagnosis'
    group_field = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/SecondPrimaryColorectalCancer/Sex'

    # Histogram
    if numeric_field in df.columns:
        plt.figure(figsize=(7,4))
        sns.histplot(df[numeric_field], bins=10, kde=True)
        plt.title('Distribution of Age at Second Diagnosis')
        plt.xlabel('Age at Second Diagnosis')
        plt.ylabel('Count')
        plt.show()

    # Boxplot by group
    if numeric_field in df.columns and group_field in df.columns:
        plt.figure(figsize=(7,4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title('Age at Second Diagnosis by Sex')
        plt.xlabel('Sex')
        plt.ylabel('Age at Second Diagnosis')
        plt.show()
else:
    print("No dataframe loaded for visualization.")

## 6. Conclusion

- We loaded and explored the **Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors** dataset using the `mlcroissant` library.
- The dataset was programmatically parsed by record set and field `@id`, supporting reproducible and robust data science workflows.
- Simple EDA showed the distribution and grouping of key numeric variables (e.g., age at second diagnosis).
- Further analysis such as modeling, stratified survival analysis, or biomarker association is possible by extending this workflow.

> **Note:** For additional context or more complex processing, consult the full Croissant schema via the dataset URL.